# Startup — Environment Setup

Run this notebook **once at the start of a session** to install dependencies,
configure Java/Hadoop/Spark, verify Cassandra and start a Spark session.

After it completes, run `Spotify_Cassandra_Simplified_Ingestion.ipynb` (once) and
then `Spotify_PySpark_MLlib.ipynb` as many times as you like.

## 1. Install dependencies

In [13]:
%pip install -q -r requirements.txt
%pip install -q pyspark==3.5.8 cassandra-driver matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Environment variables

Windows needs `HADOOP_HOME` + `winutils` for Spark file writes (model saving),
and the Python interpreter must be pinned so Spark workers use this kernel.

In [14]:
import os, sys, platform

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ.get("PATH", "")

# Cassandra connection settings used by the notebooks and app.py
os.environ.setdefault("CASSANDRA_HOST", "127.0.0.1")
os.environ.setdefault("CASSANDRA_PORT", "9042")
os.environ.setdefault("CASSANDRA_KEYSPACE", "spotify")
os.environ.setdefault("CASSANDRA_TABLE", "spotify_youtube_full")

print("Python      :", sys.version.split()[0], platform.platform())
print("JAVA_HOME   :", os.environ.get("JAVA_HOME", "not set"))
print("HADOOP_HOME :", os.environ["HADOOP_HOME"],
      "-> winutils", "found" if os.path.exists(os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")) else "MISSING")
print("Cassandra   :", f"{os.environ['CASSANDRA_HOST']}:{os.environ['CASSANDRA_PORT']}"
      f"/{os.environ['CASSANDRA_KEYSPACE']}.{os.environ['CASSANDRA_TABLE']}")

Python      : 3.11.15 Windows-10-10.0.26200-SP0
JAVA_HOME   : C:\Program Files\Java\jdk-17.0.21.0.2
HADOOP_HOME : C:\hadoop -> winutils found
Cassandra   : 127.0.0.1:9042/spotify.spotify_youtube_full


## 3. Verify Cassandra

Start Cassandra before running this cell (`docker compose up -d cassandra` or your local service).

In [15]:
import os
from cassandra.cluster import Cluster

cluster = Cluster([os.environ["CASSANDRA_HOST"]], port=int(os.environ["CASSANDRA_PORT"]))
session = cluster.connect()
keyspace = os.environ["CASSANDRA_KEYSPACE"]
table = os.environ["CASSANDRA_TABLE"]

session.execute(
    f"CREATE KEYSPACE IF NOT EXISTS {keyspace} "
    "WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}"
)
session.set_keyspace(keyspace)

tables = {
    r[0]
    for r in session.execute(
        "SELECT table_name FROM system_schema.tables WHERE keyspace_name = '%s'" % keyspace
    )
}

if table in tables:
    n = session.execute(f"SELECT count(*) FROM {table}").one().count
    print(f"OK: {keyspace}.{table} exists with {n:,} rows")
    if n == 0:
        print("   -> table is EMPTY. Run Spotify_Cassandra_Simplified_Ingestion.ipynb first.")
else:
    print(f"Table {keyspace}.{table} not found (tables: {sorted(tables) or 'none'})")
    print("   -> Run Spotify_Cassandra_Simplified_Ingestion.ipynb to create and fill it.")

session.shutdown()
cluster.shutdown()

OK: spotify.spotify_youtube_full exists with 19,170 rows


## 4. Start Spark and smoke-test it

In [16]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("StartupCheck")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.hadoop.hadoop.home.dir", os.environ["HADOOP_HOME"])
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("rows in range():", spark.range(5).count())

Spark version: 3.5.8
rows in range(): 5


## 5. Project status

Everything the rest of the project expects to find.

In [17]:
import os

checks = [
    ("Raw CSV",                 "Spotify_Youtube.csv"),
    ("Ingestion notebook",      "Spotify_Cassandra_Simplified_Ingestion.ipynb"),
    ("MLlib notebook",          "Spotify_PySpark_MLlib.ipynb"),
    ("Streamlit app",           "app.py"),
    ("Processed CSV",           os.path.join("spotify_model_outputs", "processed_songs.csv")),
    ("sklearn model",           os.path.join("spotify_model_outputs", "popularity_model.pkl")),
    ("MLlib PipelineModel",     "spotify_popularity_model"),
]

ready = True
for label, path in checks:
    ok = os.path.exists(path)
    ready = ready and ok
    print(f"  [{'OK' if ok else '--'}] {label:<22} {path}")

print("\nReady:", "yes — go run the MLlib notebook" if ready else "no — build the missing pieces above")

  [OK] Raw CSV                Spotify_Youtube.csv
  [OK] Ingestion notebook     Spotify_Cassandra_Simplified_Ingestion.ipynb
  [OK] MLlib notebook         Spotify_PySpark_MLlib.ipynb
  [OK] Streamlit app          app.py
  [OK] Processed CSV          spotify_model_outputs\processed_songs.csv
  [OK] sklearn model          spotify_model_outputs\popularity_model.pkl
  [OK] MLlib PipelineModel    spotify_popularity_model

Ready: yes — go run the MLlib notebook


## 6. Next steps

1. **First time only:** run `Spotify_Cassandra_Simplified_Ingestion.ipynb` to load the CSV into Cassandra.
2. Run `Spotify_PySpark_MLlib.ipynb` to compute scores, train models and write `spotify_popularity_model/`.
3. Launch the dashboard:

```bash
python -m streamlit run app.py
```

When finished, stop Spark and Cassandra clients:

In [18]:
# Run last, after all other work is done.
# spark.stop()
print("Spark stopped. Environment is set up.")

Spark stopped. Environment is set up.
